# 03 · Feature identification, SOH validation and temporal alignment

**Goal:**
1. Identify the cycling features documented in `DATA_DICTIONARY.csv` (the handover describes *22*) and reconcile them with the observed files.
2. Re-derive `SOH_Percent = Capacity_Ah / Reference_Capacity_Ah * 100` and confirm `Reference_Capacity_Ah` is the **first observed RPT capacity** per battery.
3. Re-derive the temporal alignment — each cycle must use only the **most recent RPT on or before its `Cycle_Date`** — and compare with what `final_ml_dataset.csv` carries.
4. Confirm `ml_training_dataset.csv` is exactly the SOH-labelled subset (no rows before the first RPT).
5. Produce a documented preprocessing table for modelling.

No generated datasets are written. Only a small JSON preprocessing table is written to `results/` (ignored by Git except for small `*.json` files).

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

# How RPTs recorded as a date only (00:00) are placed relative to cycles on the same day:
#   "start"   - as stored: the RPT applies from 00:00, so later cycles that day use it (handover rule)
#   "end"     - the RPT applies from the end of its day, so same-day cycles use the previous RPT
#   "unknown" - same-day cycles are flagged as ambiguous and not judged
# Section 6.1 compares all three; see docs/DECISIONS.md.
DATE_ONLY_RPT = "start"

## 2. Load inputs

In [ ]:
REQUIRED = [
    "DATA_DICTIONARY.csv",
    "final_battery_cycling_features.csv",
    "final_battery_health_dataset.csv",
    "rpt_capacity_data.csv",
    "final_ml_dataset.csv",
    "ml_training_dataset.csv",
]
dp.check_data_files(DATA_ROOT, required=REQUIRED)

dd = dp.load_data_dictionary(DATA_ROOT / "DATA_DICTIONARY.csv")
cycling = dp.read_dataset(DATA_ROOT / "final_battery_cycling_features.csv")
health = dp.read_dataset(DATA_ROOT / "final_battery_health_dataset.csv")
rpt = dp.read_dataset(DATA_ROOT / "rpt_capacity_data.csv")
final_ml = dp.read_dataset(DATA_ROOT / "final_ml_dataset.csv")
ml_train = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
for name, df in [("dictionary", dd), ("cycling", cycling), ("health", health), ("rpt", rpt), ("final_ml", final_ml), ("ml_training", ml_train)]:
    print(f"{name:12s} shape={df.shape}")

## 3. Identify the cycling features

Each documented or observed column is classified as:
* **identifier/time** — `Battery_ID`, `Cycling_Test`, `Cycle_Date` (used for grouping and ordering, not as inputs);
* **health/label (RPT join)** — `RPT`, `RPT_Number`, `Capacity_Ah`, `Reference_Capacity_Ah`, `SOH_Percent`, `Days_Since_RPT` (targets or target-derived → excluded from inputs to avoid leakage);
* **cycling feature** — everything else, including `Cycle_Index` (cycle count).

The handover describes "22 cycling features". The count is computed from the files, not assumed.

In [ ]:
feature_roles = dp.identify_cycling_features(dd["column_name"], cycling.columns)
display(feature_roles)

documented_features = feature_roles.index[(feature_roles["role"] == "cycling feature") & feature_roles["in_dictionary"]].tolist()
file_features = feature_roles.index[(feature_roles["role"] == "cycling feature") & feature_roles["in_cycling_features_file"]].tolist()
CYCLING_FEATURES = [c for c in documented_features if c in file_features]

print(f"columns in final_battery_cycling_features.csv : {cycling.shape[1]}")
print(f"cycling features documented in dictionary     : {len(documented_features)}")
print(f"cycling features present in the features file : {len(file_features)}")
print(f"documented AND present (used downstream)       : {len(CYCLING_FEATURES)}")
print(f"documented + identifier/time columns           : {len(documented_features) + int((feature_roles['role'] == 'identifier/time').sum())}")
print("matches the stated 22?                        :", len(CYCLING_FEATURES) == 22)
print("undocumented features in file:", [c for c in file_features if c not in documented_features] or "none")
print("documented features missing from file:", [c for c in documented_features if c not in file_features] or "none")

### 3.1 Cross-file consistency of feature values

`final_ml_dataset.csv` should carry the same feature values as `final_battery_cycling_features.csv` for the same cycle. The two are joined on `(Battery_ID, Cycling_Test, Cycle_Index)` and compared column by column (NaN on both sides counts as equal).

In [ ]:
KEY = [c for c in (dp.GROUP_COL, "Cycling_Test", "Cycle_Index") if c in cycling and c in final_ml]
shared = [c for c in CYCLING_FEATURES + ["Cycle_Date"] if c in final_ml and c not in KEY]
joined = cycling[KEY + shared].merge(final_ml[KEY + shared], on=KEY, how="outer", suffixes=("_feat", "_ml"), indicator=True)
print("join coverage:", joined["_merge"].value_counts().to_dict())
both = joined[joined["_merge"] == "both"]
rows = []
for c in shared:
    a, b = both[f"{c}_feat"], both[f"{c}_ml"]
    if pd.api.types.is_numeric_dtype(a):
        diff = (a - b).abs()
        mismatch = int((~(np.isclose(a, b, equal_nan=True))).sum())
        rows.append({"column": c, "mismatches": mismatch, "max_abs_diff": diff.max()})
    else:
        rows.append({"column": c, "mismatches": int((~((a == b) | (a.isna() & b.isna()))).sum()), "max_abs_diff": np.nan})
feature_consistency = pd.DataFrame(rows).set_index("column")
feature_consistency

## 4. Validate the SOH formula

Checked on every row where `Capacity_Ah`, `Reference_Capacity_Ah` and `SOH_Percent` are all present (absolute tolerance 1e-6 percentage points).

In [ ]:
soh_checks = pd.DataFrame({
    "health": dp.validate_soh_formula(health),
    "final_ml": dp.validate_soh_formula(final_ml),
    "ml_training": dp.validate_soh_formula(ml_train),
}).T
soh_checks

## 5. Validate `Reference_Capacity_Ah` = first observed RPT capacity

Per battery, RPTs are ordered by `Test_Date` (tie-break `RPT_Number`) and the first `Capacity_Ah` is compared with the single `Reference_Capacity_Ah` value. The RPT_Number-first view is shown alongside so any disagreement between numbering and dates is visible. We also check that `rpt_capacity_data.csv` agrees with the health dataset and that the cycle-level files use the same reference per battery.

In [ ]:
ref_check = dp.validate_reference_capacity(health)
display(ref_check)

rpt_key = [c for c in (dp.GROUP_COL, "RPT") if c in rpt and c in health]
rpt_vs_health = rpt.merge(health, on=rpt_key, how="outer", suffixes=("_rpt", "_health"), indicator=True)
print("rpt_capacity_data vs health join:", rpt_vs_health["_merge"].value_counts().to_dict())
for col in ("Capacity_Ah", "Test_Date"):
    if f"{col}_rpt" in rpt_vs_health:
        a, b = rpt_vs_health[f"{col}_rpt"], rpt_vs_health[f"{col}_health"]
        eq = np.isclose(a, b) if col == "Capacity_Ah" else (a == b)
        print(f"  {col} disagreements:", int((~pd.Series(eq, index=a.index) & a.notna() & b.notna()).sum()))

cycle_refs = pd.concat({
    name: df.groupby(df[dp.GROUP_COL].astype(str))["Reference_Capacity_Ah"].agg(["nunique", "first"])
    for name, df in [("final_ml", final_ml), ("ml_training", ml_train)]
}, axis=1)
cycle_refs[("health", "reference")] = ref_check["Reference_Capacity_Ah"]
cycle_refs[("check", "same_reference")] = np.isclose(cycle_refs[("final_ml", "first")], cycle_refs[("health", "reference")]) & np.isclose(
    cycle_refs[("ml_training", "first")], cycle_refs[("health", "reference")]
)
cycle_refs

## 6. Validate RPT temporal alignment

The alignment is recomputed independently with a backward as-of join (`pd.merge_asof(..., direction="backward", allow_exact_matches=True)`) per battery: each cycle receives the latest RPT whose `Test_Date <= Cycle_Date`. We then compare against the dataset:

| check | meaning |
|---|---|
| `rpt_match` | dataset `RPT` equals the recomputed RPT (both missing = match) |
| `soh_match` | dataset `SOH_Percent` equals that RPT's SOH |
| `days_nonnegative` | `Days_Since_RPT >= 0` |
| `days_match` | `Days_Since_RPT` within 1 day of the recomputed gap (whole vs fractional days) |
| `no_future_rpt` | the RPT the dataset assigned is not dated after the cycle (no future leakage) |

In [ ]:
align_summary_final, align_final = dp.validate_rpt_alignment(final_ml, health, date_only_rpt=DATE_ONLY_RPT)
align_summary_train, align_train = dp.validate_rpt_alignment(ml_train, health, date_only_rpt=DATE_ONLY_RPT)
display(pd.DataFrame({"final_ml": align_summary_final, "ml_training": align_summary_train}))
print("Per battery — final_ml_dataset.csv")
display(align_final)
print("Per battery — ml_training_dataset.csv")
align_train

### 6.1 Same-day RPTs: sensitivity to the date-only convention

If `Test_Date` is stored without a time of day, pandas reads it as 00:00, so a cycle later that day is matched to that RPT (`"start"`). Whether a date-only RPT really marks the start of the day, the end of the day, or an unknown time is not documented in the handover.

Only cycles on the calendar day of a date-only RPT are affected. The tables below show how many there are and how the alignment checks change under each convention. The convention under which the stored data passes is the one the handover pipeline used; it still needs to be confirmed as the physically correct one.

In [ ]:
test_dates = health["Test_Date"].dropna()
print(f"RPTs stored as date only: {int(dp.is_date_only(test_dates).sum())} of {len(test_dates)}")

same_day = dp.same_day_as_date_only_rpt(final_ml, health)
display(same_day.groupby(final_ml[dp.GROUP_COL].astype(str)).agg(same_day_cycles="sum", share_pct=lambda s: round(s.mean() * 100, 2)))

convention_rows = {}
for conv in dp.DATE_ONLY_RPT_CONVENTIONS:
    s_final, _ = dp.validate_rpt_alignment(final_ml, health, date_only_rpt=conv)
    s_train, _ = dp.validate_rpt_alignment(ml_train, health, date_only_rpt=conv)
    subset = dp.validate_training_subset(ml_train, final_ml, health, date_only_rpt=conv)
    convention_rows[conv] = {
        "final_ml_rows_before_first_rpt": s_final["rows_before_first_rpt"],
        "final_ml_rpt_match_failures": s_final.get("rpt_match_failures"),
        "final_ml_no_future_rpt_failures": s_final.get("no_future_rpt_failures"),
        "ml_training_rpt_match_failures": s_train.get("rpt_match_failures"),
        "training_subset_ok": bool(subset["ok"].all()),
        "all_alignment_checks_pass": s_final["ok"] and s_train["ok"],
    }
convention_comparison = pd.DataFrame(convention_rows).T
convention_comparison.index.name = "date_only_rpt"
convention_comparison

## 7. `ml_training_dataset.csv` is the SOH-labelled subset

Per battery:
* `train_rows_before_first_rpt` must be 0 (rows before the first RPT excluded);
* `train_missing_soh` must be 0;
* `final_pre_rpt_rows_with_soh` must be 0 (pre-RPT SOH is *missing*, not fabricated, in `final_ml_dataset.csv`);
* `train_rows == final_rows_with_soh` (nothing else dropped).

In [ ]:
subset_check = dp.validate_training_subset(ml_train, final_ml, health, date_only_rpt=DATE_ONLY_RPT)
subset_check

## 8. Preprocessing table

One row per column of `ml_training_dataset.csv`, describing its role and how the baseline (notebook 04) treats it. Statistics are computed on the full labelled file here for documentation; **any fitted preprocessing (imputation medians, scaling) is fitted on the training split only** in notebook 04.

In [ ]:
MODEL_INPUTS = dp.model_feature_columns(ml_train)

def handling(col: str) -> str:
    if col == "SOH_Percent":
        return "TARGET (regression)"
    if col in dp.HEALTH_COLUMNS:
        return "EXCLUDED — label/RPT-derived, would leak the target"
    if col == dp.GROUP_COL:
        return "Grouping key — split and evaluate per battery; not an input"
    if col == "Cycle_Date":
        return "Ordering key — chronological split; not an input"
    if col == "Cycling_Test":
        return "Segment identifier — ordering only; not an input"
    if col in dp.TEMPERATURE_COLUMNS:
        return "INPUT — missing kept as NaN; median (train-only) + missing indicator for linear model; native NaN for tree model"
    if col in MODEL_INPUTS:
        return "INPUT — numeric, standardised (train-only) for linear model"
    return "Not used (non-numeric)"

preprocessing_table = pd.DataFrame({
    "role": [feature_roles["role"].get(c, "undocumented") for c in ml_train.columns],
    "in_dictionary": [c in set(dd["column_name"]) for c in ml_train.columns],
    "dtype": ml_train.dtypes.astype(str).values,
    "missing": ml_train.isna().sum().values,
    "missing_pct": (ml_train.isna().mean() * 100).round(3).values,
    "model_input": [c in MODEL_INPUTS for c in ml_train.columns],
    "handling": [handling(c) for c in ml_train.columns],
}, index=pd.Index(ml_train.columns, name="column"))
display(preprocessing_table)
print(f"{len(MODEL_INPUTS)} model inputs:", MODEL_INPUTS)

In [ ]:
RESULTS_DIR.mkdir(exist_ok=True)
out_path = RESULTS_DIR / "preprocessing_table.json"
preprocessing_table.reset_index().to_json(out_path, orient="records", indent=2)
print("wrote", out_path.relative_to(REPO_ROOT), f"({out_path.stat().st_size} bytes)")

## 9. Summary

In [ ]:
validation_summary = pd.DataFrame(
    [
        ("Cycling features identified", len(CYCLING_FEATURES), f"{len(CYCLING_FEATURES)} (handover states 22)"),
        ("Feature values consistent between cycling file and final_ml", int(feature_consistency["mismatches"].sum()) == 0,
         f"{int(feature_consistency['mismatches'].sum())} mismatching cells"),
        ("SOH formula holds (health / final_ml / ml_training)", bool(soh_checks["ok"].all()),
         f"mismatches: {soh_checks['mismatches'].to_dict()}"),
        ("Reference = first observed RPT capacity", bool(ref_check["ok"].all()),
         f"failing batteries: {ref_check.index[~ref_check['ok']].tolist()}"),
        ("Cycle-level files use the same reference as health", bool(cycle_refs[("check", "same_reference")].all()), ""),
        ("Date-only RPT convention used", DATE_ONLY_RPT,
         f"{int(same_day.sum())} cycles on a date-only RPT day; passing conventions: "
         f"{convention_comparison.index[convention_comparison['all_alignment_checks_pass'].astype(bool)].tolist()}"),
        ("final_ml aligned to most recent RPT on/before cycle", align_summary_final["ok"],
         {k: v for k, v in align_summary_final.items() if k.endswith("_failures")}),
        ("ml_training aligned to most recent RPT on/before cycle", align_summary_train["ok"],
         {k: v for k, v in align_summary_train.items() if k.endswith("_failures")}),
        ("ml_training excludes pre-RPT rows and equals labelled subset", bool(subset_check["ok"].all()),
         f"failing batteries: {subset_check.index[~subset_check['ok']].tolist()}"),
        ("final_ml pre-RPT rows (SOH missing, not fabricated)", int(subset_check["final_rows_before_first_rpt"].sum()), ""),
        ("Model inputs (leakage columns excluded)", len(MODEL_INPUTS), ""),
    ],
    columns=["check", "result", "detail"],
)
validation_summary

### Unresolved assumptions

* **Feature count.** The modelling code uses the list derived in section 3 (18 input features under the handover dictionary), not a hard-coded count. The README's earlier "22 features" wording is pending confirmation with the data owner (see `docs/DECISIONS.md`).
* **Date-only RPTs.** The on-or-before rule is evaluated with `DATE_ONLY_RPT` (default `"start"`, i.e. timestamps as stored). Section 6.1 shows which conventions the stored data is consistent with and how many cycles are affected; the physically correct convention must be confirmed with the data owner (see `docs/DECISIONS.md`).
* **`Days_Since_RPT` convention.** Accepted within ±1 day of the recomputed fractional gap (whole-day truncation vs. fractional days).
* **`Days_Since_RPT` and `Cycle_Index` as inputs.** `Days_Since_RPT` is excluded as RPT-derived. `Cycle_Index` is kept (cycle count is a documented feature) but it is strongly correlated with time and therefore with SOH; it may reset between `Cycling_Test` segments.
* **Temperature handling** is deferred to the model: native NaN support for tree models, train-fitted median + indicator for linear models, masking for the TFT.